# AgriMatch M2 -- CHIRPS Rainfall Backfill

> **Optimised for Colab L4 GPU runtime**

This notebook downloads CHIRPS v2.0 daily rainfall data for all 260 Ghana
districts from **2006-01-01 to 2023-07-15** and saves the results as
`chirps_daily.csv` for import into your local PostgreSQL database.

## Before you start

- **Do not close this browser tab while running.** Colab will disconnect
  if the tab is closed or the session times out.
- **Runtime**: 2-3 hours on a standard Colab instance with 12 parallel
  download workers.
- **Restartable**: progress is saved to `/content/chirps_progress.csv`
  every 100 dates. If the session disconnects, re-run all cells --
  completed dates will be skipped automatically.
- **Output**: `chirps_daily.csv` (~6,400 dates x 260 districts = ~1.6M
  rows). The file is downloaded automatically after the final phase.

## What this produces

| Column | Description |
|---|---|
| `obs_date` | Observation date (YYYY-MM-DD) |
| `district_name` | GADM NAME_2 district name |
| `region_name` | GADM NAME_1 region name |
| `mean_rainfall_mm` | Mean daily rainfall across district pixels (mm) |
| `cell_count` | Number of 0.05-degree pixels aggregated |


In [ ]:
# Cell 2 -- Install dependencies
!pip install rasterio geopandas requests --quiet

In [ ]:
# Cell 3 -- Imports
import gzip
import json
import logging
import os
import shutil
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, timedelta
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rasterio.transform as rtransform
import requests
from rasterio.windows import from_bounds
from shapely.geometry import Point

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s -- %(message)s",
    datefmt="%H:%M:%S",
)

print("Imports OK")

In [ ]:
# Cell 4 -- Ghana bounding box and district setup

# Ghana bounding box (WGS84)
GHA_MIN_LON = -3.25
GHA_MAX_LON =  1.20
GHA_MIN_LAT =  4.50
GHA_MAX_LAT = 11.20

GADM_URL  = "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_GHA.gpkg"
GADM_PATH = Path("/content/gadm41_GHA.gpkg")


def load_ghana_districts() -> gpd.GeoDataFrame:
    """Download GADM Ghana level-2 boundaries and return as GeoDataFrame."""
    if not GADM_PATH.exists():
        print(f"Downloading GADM Ghana boundaries (~25 MB) ...")
        resp = requests.get(GADM_URL, stream=True, timeout=(15, 300))
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        downloaded = 0
        with GADM_PATH.open("wb") as fh:
            for chunk in resp.iter_content(chunk_size=65536):
                if chunk:
                    fh.write(chunk)
                    downloaded += len(chunk)
        print(f"  Downloaded {downloaded / 1_048_576:.1f} MB -> {GADM_PATH}")
    else:
        print(f"GADM file already present: {GADM_PATH}")

    gdf = gpd.read_file(GADM_PATH, layer="ADM_ADM_2")

    gdf = gdf[["GID_2", "NAME_1", "NAME_2", "geometry"]].copy()
    gdf = gdf.rename(columns={"NAME_1": "region_name", "NAME_2": "district_name"})
    gdf = gdf.to_crs("EPSG:4326")

    centroids = gdf.geometry.centroid
    gdf["centroid_lat"] = centroids.y.round(6)
    gdf["centroid_lon"] = centroids.x.round(6)

    # Sequential district_id 1..N sorted by GID_2 for stable ordering
    gdf = gdf.sort_values("GID_2").reset_index(drop=True)
    gdf["district_id"] = gdf.index + 1

    print(f"Loaded {len(gdf)} districts across {gdf['region_name'].nunique()} regions")
    print(gdf[["district_id", "district_name", "region_name"]].head(5).to_string(index=False))
    return gdf


gdf_districts = load_ghana_districts()

In [ ]:
# Cell 5 -- CHIRPS download and processing functions

_CHIRPS_BASE = (
    "https://data.chc.ucsb.edu/products/CHIRPS-2.0"
    "/global_daily/tifs/p05"
)
_NODATA = -9999.0
_TMP_DIR = Path("/content/chirps_tmp")
_TMP_DIR.mkdir(parents=True, exist_ok=True)


def get_download_url(target_date: date) -> str:
    date_str = target_date.strftime("%Y.%m.%d")
    return f"{_CHIRPS_BASE}/{target_date.year}/chirps-v2.0.{date_str}.tif.gz"


def download_and_extract(target_date: date, tmp_dir: Path = _TMP_DIR) -> Path:
    """Download CHIRPS .tif.gz and decompress to .tif. Returns tif path."""
    url     = get_download_url(target_date)
    stamp   = target_date.strftime("%Y%m%d")
    gz_path  = tmp_dir / f"chirps_{stamp}.tif.gz"
    tif_path = tmp_dir / f"chirps_{stamp}.tif"

    if tif_path.exists():
        return tif_path

    for attempt in range(2):
        resp = requests.get(url, stream=True, timeout=(10, 300))
        if resp.status_code in (429, 503) and attempt == 0:
            time.sleep(60)
            continue
        resp.raise_for_status()
        break

    with gz_path.open("wb") as fh:
        for chunk in resp.iter_content(chunk_size=65536):
            if chunk:
                fh.write(chunk)

    with gzip.open(gz_path, "rb") as gz_in, tif_path.open("wb") as tif_out:
        shutil.copyfileobj(gz_in, tif_out)

    gz_path.unlink(missing_ok=True)
    return tif_path


def extract_and_aggregate(
    tif_path: Path,
    target_date: date,
    gdf_districts: gpd.GeoDataFrame,
) -> pd.DataFrame:
    """
    Clip raster to Ghana bbox, spatial-join to districts, aggregate mean
    rainfall per district.

    Returns DataFrame with columns:
        obs_date, district_name, region_name, mean_rainfall_mm, cell_count
    """
    with rasterio.open(tif_path) as src:
        window = from_bounds(
            GHA_MIN_LON, GHA_MIN_LAT, GHA_MAX_LON, GHA_MAX_LAT,
            src.transform,
        )
        data          = src.read(1, window=window)
        win_transform = src.window_transform(window)

    valid_mask = (data != _NODATA) & (data >= 0)
    rows, cols = np.where(valid_mask)

    if rows.size == 0:
        return pd.DataFrame(
            columns=["obs_date", "district_name", "region_name",
                     "mean_rainfall_mm", "cell_count"]
        )

    lons, lats = rtransform.xy(win_transform, rows, cols)
    rainfall   = data[rows, cols].astype(float)

    gdf_cells = gpd.GeoDataFrame(
        {
            "rainfall_mm": rainfall,
            "geometry":    [Point(lon, lat) for lon, lat in zip(lons, lats)],
        },
        crs="EPSG:4326",
    )

    joined = gpd.sjoin(
        gdf_cells,
        gdf_districts[["district_name", "region_name", "geometry"]],
        how="inner",
        predicate="within",
    )

    agg = (
        joined.groupby(["district_name", "region_name"])["rainfall_mm"]
        .agg(mean_rainfall_mm="mean", cell_count="count")
        .reset_index()
    )
    agg["obs_date"]         = target_date.isoformat()
    agg["mean_rainfall_mm"] = agg["mean_rainfall_mm"].round(3)

    return agg[["obs_date", "district_name", "region_name",
                "mean_rainfall_mm", "cell_count"]]


print("CHIRPS functions defined OK")

In [ ]:
# Cell 6 -- Parallel backfill function

PROGRESS_CSV = Path("/content/chirps_progress.csv")
SAVE_EVERY   = 100  # dates between progress saves


def _process_one_date(
    target_date: date,
    gdf_districts: gpd.GeoDataFrame,
) -> tuple[date, pd.DataFrame | None, str | None]:
    """Download, extract, aggregate one date. Returns (date, df, error_msg)."""
    tif_path = None
    try:
        tif_path = download_and_extract(target_date)
        df       = extract_and_aggregate(tif_path, target_date, gdf_districts)
        return target_date, df, None
    except Exception as exc:
        return target_date, None, str(exc)
    finally:
        if tif_path and tif_path.exists():
            tif_path.unlink(missing_ok=True)


def run_backfill(
    start_date: date,
    end_date:   date,
    max_workers: int = 12,
) -> pd.DataFrame:
    """
    Download and aggregate CHIRPS data for every date in [start_date, end_date].

    Skips dates already present in the progress CSV.
    Saves progress to PROGRESS_CSV every SAVE_EVERY completed dates.
    Returns combined DataFrame of all results.
    """
    # Build full date list
    all_dates: list[date] = []
    d = start_date
    while d <= end_date:
        all_dates.append(d)
        d += timedelta(days=1)

    # Load existing progress
    if PROGRESS_CSV.exists():
        existing = pd.read_csv(PROGRESS_CSV, parse_dates=["obs_date"])
        existing["obs_date"] = existing["obs_date"].dt.date
        done_dates = set(existing["obs_date"].unique())
        accumulated = [existing]
        print(f"Resuming: {len(done_dates)} dates already in progress CSV")
    else:
        done_dates  = set()
        accumulated = []

    missing = [d for d in all_dates if d not in done_dates]
    total   = len(all_dates)
    print(
        f"Date range {start_date} to {end_date}: "
        f"{total} total, {len(done_dates)} done, {len(missing)} to fetch"
    )

    if not missing:
        print("Nothing to fetch.")
        return pd.concat(accumulated, ignore_index=True) if accumulated else pd.DataFrame()

    failed_dates: list[date] = []
    completed   = 0
    t_start     = time.time()
    new_frames: list[pd.DataFrame] = []

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {
            pool.submit(_process_one_date, d, gdf_districts): d
            for d in missing
        }
        for future in as_completed(futures):
            target_date, df, err = future.result()
            completed += 1

            if err:
                print(f"  FAILED {target_date}: {err}")
                failed_dates.append(target_date)
            elif df is not None and not df.empty:
                new_frames.append(df)

            # Save progress every SAVE_EVERY dates
            if completed % SAVE_EVERY == 0 and new_frames:
                checkpoint = pd.concat(accumulated + new_frames, ignore_index=True)
                checkpoint.to_csv(PROGRESS_CSV, index=False)
                print(f"  [checkpoint] Saved {len(checkpoint)} rows to progress CSV")

            # Print progress every 50 dates
            if completed % 50 == 0 or completed == len(missing):
                elapsed  = time.time() - t_start
                rate     = completed / elapsed if elapsed > 0 else 0
                rem      = (len(missing) - completed) / rate if rate > 0 else 0
                total_done = len(done_dates) + completed
                print(
                    f"  Completed {total_done}/{total} dates | "
                    f"Elapsed: {elapsed/60:.1f}m | "
                    f"Est remaining: {rem/60:.1f}m"
                )

    # Final save
    all_frames = accumulated + new_frames
    result = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
    result.to_csv(PROGRESS_CSV, index=False)
    print(f"Progress saved: {len(result)} rows in {PROGRESS_CSV}")

    if failed_dates:
        print(f"\nFailed dates ({len(failed_dates)}):")
        for fd in sorted(failed_dates):
            print(f"  {fd}")

    return result


print("run_backfill defined OK")


In [ ]:
# Cell 7 -- Run the backfill in three phases

OUTPUT_CSV = Path("/content/chirps_daily.csv")

PHASES = [
    ("Phase 1 -- Tier 1 history",    date(2006, 1, 1),  date(2018, 12, 31)),
    ("Phase 2 -- Tier 3 + recent",   date(2019, 1, 1),  date(2022, 12, 31)),
    ("Phase 3 -- 2023 to price data", date(2023, 1, 1), date(2023, 7, 15)),
]

for phase_num, (label, start, end) in enumerate(PHASES, 1):
    print()
    print("=" * 65)
    print(f"[Phase {phase_num}] {label}")
    print(f"  {start} to {end}")
    print("=" * 65)

    phase_df = run_backfill(start, end, max_workers=12)

    # Merge all phases into output CSV (re-read progress CSV which has all phases)
    if PROGRESS_CSV.exists():
        combined = pd.read_csv(PROGRESS_CSV)
        combined.to_csv(OUTPUT_CSV, index=False)

    print()
    print(f"Phase {phase_num} complete:")
    if phase_df is not None and not phase_df.empty:
        print(f"  Rows in phase result : {len(phase_df):,}")
        print(f"  Date range           : {phase_df['obs_date'].min()} to {phase_df['obs_date'].max()}")
    if OUTPUT_CSV.exists():
        combined = pd.read_csv(OUTPUT_CSV)
        print(f"  Total rows in CSV    : {len(combined):,}")
        print(f"  Full date range      : {combined['obs_date'].min()} to {combined['obs_date'].max()}")

print()
print("All phases complete. chirps_daily.csv is ready.")


In [ ]:
# Cell 8 -- Validation and download

if not OUTPUT_CSV.exists():
    print("ERROR: chirps_daily.csv not found. Run Cell 7 first.")
else:
    df = pd.read_csv(OUTPUT_CSV)

    print()
    print("=" * 65)
    print("FINAL SUMMARY -- chirps_daily.csv")
    print("=" * 65)
    print(f"  Total rows           : {len(df):,}")
    print(f"  Date range           : {df['obs_date'].min()} to {df['obs_date'].max()}")
    print(f"  Unique dates         : {df['obs_date'].nunique():,}")
    print(f"  Unique districts     : {df['district_name'].nunique()}")
    print(f"  Rainfall range (mm)  : {df['mean_rainfall_mm'].min():.3f} to {df['mean_rainfall_mm'].max():.3f}")
    print()

    # Dates with fewer than 200 districts (coverage warning)
    dist_per_date = df.groupby("obs_date")["district_name"].nunique()
    low_coverage  = dist_per_date[dist_per_date < 200]
    if len(low_coverage) > 0:
        print(f"  WARNING: {len(low_coverage)} dates have fewer than 200 districts")
        print(low_coverage.head(10).to_string())
    else:
        print(f"  Coverage OK: all dates have 200+ districts")

    # Failed dates = dates in expected range but missing from CSV
    all_expected: list[str] = []
    d = date(2006, 1, 1)
    while d <= date(2023, 7, 15):
        all_expected.append(d.isoformat())
        d += timedelta(days=1)

    present  = set(df["obs_date"].unique())
    missing  = sorted(set(all_expected) - present)

    print()
    if missing:
        print(f"  Missing dates ({len(missing)}) -- retry with:")
        print("    from datetime import date")
        print("    run_backfill(date(YYYY, MM, DD), date(YYYY, MM, DD))")
        for md in missing[:20]:
            print(f"    {md}")
        if len(missing) > 20:
            print(f"    ... and {len(missing) - 20} more")
    else:
        print("  No missing dates -- full range covered.")

    print()
    print("Downloading chirps_daily.csv ...")
    from google.colab import files
    files.download("/content/chirps_daily.csv")

## Importing into local PostgreSQL

After downloading `chirps_daily.csv`, run the following script from your
project root to import the data into `chirps_daily`.

The script matches rows by `district_name` against `ghana_districts.district_name`
and skips any rows that cannot be matched or already exist in the table.

### Step 1 -- Copy the CSV to your project

Move `chirps_daily.csv` into your project root (or any path you prefer).

### Step 2 -- Run the import script

Save the following as `setup/import_chirps_csv.py` and run it:

```python
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).parent.parent))

import pandas as pd
from db.connection import get_session
from sqlalchemy import text

CSV_PATH = Path("chirps_daily.csv")  # adjust path if needed

df = pd.read_csv(CSV_PATH, parse_dates=["obs_date"])
df["obs_date"] = df["obs_date"].dt.date
print(f"Loaded {len(df):,} rows from {CSV_PATH}")

# Resolve district_name -> district_id
with get_session() as s:
    id_rows = s.execute(text(
        "SELECT id, district_name FROM ghana_districts"
    )).all()
name_to_id = {r.district_name: r.id for r in id_rows}

df["district_id"] = df["district_name"].map(name_to_id)
unmatched = df[df["district_id"].isna()]["district_name"].unique()
if len(unmatched) > 0:
    print(f"WARNING: {len(unmatched)} district names did not match:")
    for n in sorted(unmatched):
        print(f"  {n}")

df = df.dropna(subset=["district_id"])
df["district_id"] = df["district_id"].astype(int)

records = df[["obs_date", "district_id", "mean_rainfall_mm", "cell_count"]]\
    .to_dict(orient="records")

BATCH = 5000
total_inserted = 0
for i in range(0, len(records), BATCH):
    batch = records[i : i + BATCH]
    with get_session() as s:
        before = s.execute(text("SELECT COUNT(*) FROM chirps_daily")).scalar()
        s.execute(text("""
            INSERT INTO chirps_daily
                (obs_date, district_id, mean_rainfall_mm, cell_count)
            VALUES
                (:obs_date, :district_id, :mean_rainfall_mm, :cell_count)
            ON CONFLICT (obs_date, district_id) DO NOTHING
        """), batch)
        after = s.execute(text("SELECT COUNT(*) FROM chirps_daily")).scalar()
    total_inserted += after - before
    print(f"  Batch {i//BATCH + 1}: inserted {after - before} rows (total {total_inserted:,})")

print(f"Import complete: {total_inserted:,} rows inserted into chirps_daily")
```

### Step 3 -- Verify

```sql
SELECT
    COUNT(*)       AS total_rows,
    MIN(obs_date)  AS date_min,
    MAX(obs_date)  AS date_max,
    COUNT(DISTINCT district_id) AS districts
FROM chirps_daily;
```

Expected: ~1.66M rows, date range 2006-01-01 to 2023-07-15, 260 districts.
